# PPQ Training

Run on the DGX (8x A100) after `ppq_data_pipeline.ipynb`. Stages:
1. Build the initial checkpoint: Qwen backbone weights + transplanted depformer.
2. Stage A â€” audio-side warmup (backbone frozen, ~6 h wall time).
3. Stage B â€” full fine-tuning (~2-3 days).
4. Evaluation (per-codebook loss + offline generation + response delays).
5. Serving with the stock PersonaPlex web UI.

In [ ]:
import os, sys, json, subprocess
from pathlib import Path

# ---- EDIT THESE ----
PPQ_REPO = "/workspace/personaplex-qwen"
MOSHI_REPO = "/workspace/personaplex"
MIMI_WEIGHT = "/workspace/models/mimi.safetensors"
QWEN_BASE = "/workspace/models/qwen3-1.7b-base"        # local Qwen3-1.7B-Base snapshot
PERSONAPLEX_CKPT = "/workspace/models/personaplex/model.sft"  # personaplex/moshiko LM checkpoint
DATA_ROOT = Path("/workspace/ppq_data")
TOKENS_DIR = DATA_ROOT / "tokens"
RUN_DIR = Path("/workspace/ppq_runs/run1")
CLIENT_DIST = Path(MOSHI_REPO) / "client" / "dist"
N_GPUS = 8
# --------------------
os.environ["PYTHONPATH"] = f"{MOSHI_REPO}/moshi:{PPQ_REPO}:" + os.environ.get("PYTHONPATH", "")
sys.path.insert(0, f"{MOSHI_REPO}/moshi")
sys.path.insert(0, PPQ_REPO)
RUN_DIR.mkdir(parents=True, exist_ok=True)

## 1. Initial checkpoint
Qwen3-1.7B-Base backbone + transplanted PersonaPlex depformer; fresh audio
embeddings, depformer_in, depformer_text_emb (trained in stage A).


In [ ]:
from ppq.config import PPQModelConfig
from ppq.checkpoint import build_model_from_scratch, save_pretrained
cfg = PPQModelConfig()
model = build_model_from_scratch(cfg, qwen_dir=QWEN_BASE, depformer_source=PERSONAPLEX_CKPT)
save_pretrained(model, RUN_DIR / 'init')
print('init checkpoint saved:', RUN_DIR / 'init')
del model


## 2. Stage A - audio-side warmup
Backbone layers, final norm and text head frozen; everything else trains.
About 6 hours for ~100 h LibriTTS-R monologues + ~100 h synth dialogues,
batch 4 x 8 GPUs, 100 s segments.


In [ ]:
import torch
from ppq.config import TrainConfig
tcfg_a = TrainConfig()
tcfg_a.data_dir = str(TOKENS_DIR)
tcfg_a.output_dir = str(RUN_DIR)
tcfg_a.stage = 'a_audio_warmup'
tcfg_a.batch_size = 4
tcfg_a.grad_accum = 1
tcfg_a.max_steps = 5000          # ~= 5000 * 32 * 100 s = 4400 h of audio
# more realistic: mix of ~200 h; reduce steps for a ~6 h run:
tcfg_a.max_steps = 400
tcfg_a.save_every = 200
tcfg_a.log_every = 10
tcfg_a.fsdp = False              # DDP; set True + fully_shard config for FSDP2
tcfg_a.gradient_checkpointing = False
tcfg_a.to_json(RUN_DIR / 'train_stage_a.json')
print(RUN_DIR / 'train_stage_a.json')


In [ ]:
# Launch stage A (DDP, 8 GPUs)
subprocess.run([
    'torchrun', '--nproc_per_node', str(N_GPUS), '-m', 'ppq.train.train',
    '--config', str(RUN_DIR / 'train_stage_a.json')], check=True)


## 3. Stage B - full fine-tuning
All weights trainable, backbone at 0.5x LR, depformer at 1.5x LR.
2-3 days for ~800 h of synthetic dialogues (plus optional stage-A data).
Resumes automatically from <output_dir>/checkpoints/last.


In [ ]:
from ppq.config import TrainConfig
tcfg_b = TrainConfig()
tcfg_b.data_dir = str(TOKENS_DIR)
tcfg_b.output_dir = str(RUN_DIR)
tcfg_b.stage = 'b_full'
tcfg_b.batch_size = 4
tcfg_b.max_steps = 3000
tcfg_b.save_every = 250
tcfg_b.log_every = 10
tcfg_b.lr = 5e-5
tcfg_b.backbone_lr_mult = 0.5
tcfg_b.depformer_lr_mult = 1.5
tcfg_b.user_codebook_weight = 0.1   # slight supervision on user streams
tcfg_b.to_json(RUN_DIR / 'train_stage_b.json')
subprocess.run([
    'torchrun', '--nproc_per_node', str(N_GPUS), '-m', 'ppq.train.train',
    '--config', str(RUN_DIR / 'train_stage_b.json')], check=True)


## 4. Evaluation


In [ ]:
import torch, glob
from torch.utils.data import DataLoader
from ppq.checkpoint import load_pretrained, get_mimi
from ppq.data.dataset import TokenShardDataset, collate
from ppq.train.evaluate import eval_per_codebook, generate_offline, write_wav, response_delays
from ppq.text_tokenizer import QwenTextTokenizer
import sphn, numpy as np
CKPT = RUN_DIR / 'checkpoints' / 'final'
device = 'cuda'
model = load_pretrained(CKPT, device=device)
mimi = get_mimi(MIMI_WEIGHT, device)
tok = QwenTextTokenizer(QWEN_BASE, model.cfg)
ds = TokenShardDataset(glob.glob(str(TOKENS_DIR / 'manifest*.jsonl')), model.cfg,
                       segment_frames=1250, seed=123, epoch_multiplier=1)
loader = DataLoader(ds, batch_size=4, collate_fn=collate, shuffle=False)
losses = eval_per_codebook(model, loader, n_batches=20, device=device, mcfg=model.cfg)
print(json.dumps({k: round(v, 3) for k, v in losses.items()}, indent=1))


In [ ]:
# Offline generation vs a recorded user utterance
user_wav, sr = sphn.read(str(DATA_ROOT / 'voice_prompt_agent.wav'))  # replace with a real user sample
if sr != 24000:
    user_wav = sphn.resample(user_wav, sr, 24000)
agent_wav, transcript = generate_offline(
    model, mimi, tok, user_wav[0], user_wav[0][: 24000 * 10],
    'You are a helpful voice assistant.', device=device, max_seconds=30.0)
write_wav(str(RUN_DIR / 'agent_reply.wav'), agent_wav)
print('transcript:', transcript)
print('delays:', response_delays(agent_wav, user_wav[0]))


## 5. Serving
Same WebSocket protocol as PersonaPlex: point the stock web UI at this server.
Voice prompts are 24 kHz .wav files in a directory; the client built-in .pt
voice entries automatically fall back to --default-voice.


In [ ]:
VOICES_DIR = DATA_ROOT / 'voices'
VOICES_DIR.mkdir(exist_ok=True)
# put agent voice wavs here, e.g. af_heart.wav = the data-pipeline voice prompt
import shutil
shutil.copy(DATA_ROOT / 'voice_prompt_agent.wav', VOICES_DIR / 'af_heart.wav')
cmd = [
    sys.executable, '-m', 'ppq.serve.server',
    '--ppq-checkpoint', str(RUN_DIR / 'checkpoints' / 'final'),
    '--mimi-weight', MIMI_WEIGHT,
    '--tokenizer', QWEN_BASE,
    '--voice-prompt-dir', str(VOICES_DIR),
    '--default-voice', 'af_heart.wav',
    '--static', str(CLIENT_DIST),
    '--host', '0.0.0.0', '--port', '8998',
]
print(' \
'.join(cmd))
# subprocess.run(cmd, check=True)  # runs forever; use a tmux session on the DGX
